In [ ]:
import numpy as np
import pandas as pd
import torch
from gcn_model import GCNModel
import utilities
from test_model import test_model
import os
import statistics
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import random

In [ ]:
data_folder = "/home/groups/ConradLab/daniel/sharp_data/sharp_sims/splat_0.7_de_rq_v3/"
# get labels
data_path = data_folder + "query_counts.csv"
tools = ["sctype","scsorter","scina","singler", "scpred"]
#tools = ["scsorter","scina","singler", "scpred"]
ref_path = data_folder + "ref_counts.csv"
ref_label_path = data_folder + "ref_labels.csv"
marker_path = data_folder + "markers.txt"
if os.path.exists(data_folder + "preds.csv"):
    all_labels = pd.read_csv(data_folder + "preds.csv", index_col=0)
    if all_labels.shape[1] != len(tools): 
        all_labels = all_labels[tools]
        #raise Exception("wrong amount of tools in file")
else:
    all_labels = utilities.label_counts(data_path,tools,ref_path,ref_label_path,marker_path)

# read in dataset
X = pd.read_csv(data_path, index_col=0)
X, keep_cells, keep_genes, pca_obj = utilities.preprocess(np.array(X), scale=False, comps=500)

all_labels = all_labels.loc[keep_cells,:]

_,marker_names = utilities.read_marker_file(marker_path)

all_labels_factored = utilities.factorize_df(all_labels, marker_names)
encoded_labels = utilities.encode_predictions(all_labels_factored)

meta_path = data_folder + "query_meta.csv"
metadata = pd.read_csv(meta_path, index_col=0)
real_y = pd.factorize(metadata['Group'], sort=True)[0]
real_y = real_y[keep_cells]

confident_labels = utilities.get_consensus_labels(encoded_labels, necessary_vote = .51)

train_nodes = np.where(confident_labels != -1)[0]
test_nodes = np.where(confident_labels == -1)[0]

In [ ]:
len(test_nodes)

In [ ]:
confusion_matrix(real_y[train_nodes], confident_labels[train_nodes])

In [ ]:
confusion_matrix(real_y, all_labels_factored["scina"])

In [ ]:
confusion_matrix(real_y, all_labels_factored["sctype"])

In [ ]:
confusion_matrix(real_y, all_labels_factored["singler"])

In [ ]:
confusion_matrix(real_y, all_labels_factored["scpred"])

In [ ]:
confusion_matrix(real_y, all_labels_factored["scsorter"])

In [ ]:
print(utilities.pred_accuracy(all_labels_factored['scina'], real_y))
print(utilities.pred_accuracy(all_labels_factored['sctype'], real_y))
print(utilities.pred_accuracy(all_labels_factored['scsorter'], real_y))
print(utilities.pred_accuracy(all_labels_factored['singler'], real_y))
print(utilities.pred_accuracy(all_labels_factored['scpred'], real_y))
max_pred = torch.tensor(encoded_labels).max(dim=1)[1]
utilities.pred_accuracy(max_pred, real_y)

In [ ]:
print(utilities.pred_accuracy(np.array(all_labels_factored['scina'][test_nodes]), real_y[test_nodes]))
print(utilities.pred_accuracy(np.array(all_labels_factored['sctype'][test_nodes]), real_y[test_nodes]))
print(utilities.pred_accuracy(np.array(all_labels_factored['scsorter'][test_nodes]), real_y[test_nodes]))
print(utilities.pred_accuracy(np.array(all_labels_factored['singler'][test_nodes]), real_y[test_nodes]))
print(utilities.pred_accuracy(np.array(all_labels_factored['scpred'][test_nodes]), real_y[test_nodes]))
max_pred = torch.tensor(encoded_labels).max(dim=1)[1]
print(utilities.pred_accuracy(max_pred[test_nodes], real_y[test_nodes]))

In [ ]:
dataset  = torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(confident_labels))
dataloader = torch.utils.data.DataLoader(dataset, batch_size=50, shuffle=True)

test_dataset  = torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(real_y))
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=50, shuffle=False)

In [ ]:
m = GCNModel("configs/2_40.txt", 2, dropout=0.0)
m.train(dataloader, 150)

In [ ]:
m.validation_metrics(test_dataloader, train_nodes, test_nodes)

In [ ]:
# get tool weights by cell
ultra_confident_labels = utilities.get_consensus_labels(encoded_labels, necessary_vote = .7)
ultra_conf_nodes = np.where(ultra_confident_labels != -1)[0]
weights = np.zeros((4,5))
for type in [0,1,2,3]:
    
    type_indices = np.where(confident_labels[ultra_conf_nodes]==type)[0]
    for i, tool in enumerate(all_labels_factored.columns):
        weights[type,i] = utilities.pred_accuracy(all_labels_factored[tool].to_numpy()[ultra_conf_nodes][type_indices], confident_labels[ultra_conf_nodes][type_indices])
weights

In [ ]:
for i, tool in enumerate(all_labels_factored.columns):
        print(utilities.pred_accuracy(all_labels_factored[tool].to_numpy()[ultra_conf_nodes], confident_labels[ultra_conf_nodes]))

In [ ]:
print(all_labels_factored.columns)

In [ ]:
new_encoded = utilities.weighted_encode(all_labels_factored, encoded_labels, weights)
confident_labels = utilities.get_consensus_labels(encoded_labels, necessary_vote = .51)

train_nodes = np.where(confident_labels != -1)[0]
test_nodes = np.where(confident_labels == -1)[0]

In [ ]:
all_labels_factored.iloc[three,:].iloc[8,:]

In [ ]:
encoded_labels[three][7]

In [ ]:
new_encoded[three][7]

In [ ]:
confident_labels[three][7]

In [ ]:
len(test_nodes)

In [ ]:
confusion_matrix(real_y[train_nodes], confident_labels[train_nodes])

In [ ]:
dataset  = torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(confident_labels))
dataloader = torch.utils.data.DataLoader(dataset, batch_size=50, shuffle=True)

test_dataset  = torch.utils.data.TensorDataset(torch.tensor(X), torch.tensor(real_y))
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=50, shuffle=False)

In [ ]:
m = GCNModel("configs/2_40.txt", 2, dropout=0.0)
m.train(dataloader, 150)

In [ ]:
m.validation_metrics(test_dataloader, train_nodes, test_nodes)

In [ ]:
# add copies of confidently labelled cell type 4 to boost numbers

In [ ]:
X.shape

In [ ]:
conf_four_cells = np.where(confident_labels == 3)[0]
X[conf_four_cells,:]

In [ ]:
repeated_four = np.tile(X[conf_four_cells,:], (2,1))
repeated_four.shape

In [ ]:
test = np.array(X[confident_labels==0,18], dtype="float64")
statistics.stdev(test)

In [ ]:
random_mat = np.zeros(repeated_four.shape)
for i in range(X.shape[1]):
    temp_sd = statistics.stdev(np.array(X[confident_labels==0,i], dtype="float64"))
    random_column = np.random.normal(0,temp_sd, repeated_four.shape[0])
    random_mat[:,i] = random_column

In [ ]:
random_mat

In [ ]:
#repeated_four = repeated_four + np.random.normal(0, 4, repeated_four.shape)
repeated_four = repeated_four + random_mat
repeated_four

In [ ]:
X_extended = np.concatenate((X, repeated_four), axis=0)
X_extended.shape

In [ ]:
confident_labels.shape

In [ ]:
extended_conf_labels = np.concatenate((confident_labels, np.array([3]*48)))
extended_conf_labels.shape

In [ ]:
extended_real_y = np.concatenate((real_y, np.array([3]*48)))
extended_real_y.shape

In [ ]:
# mix in fake nodes
random.seed(8)
shuffled = list(range(len(extended_real_y)))
random.shuffle(shuffled)
extended_real_y = extended_real_y[shuffled]
extended_conf_labels = extended_conf_labels[shuffled]
X_extended = X_extended[shuffled,:]
train_nodes = np.where(extended_conf_labels != -1)[0]
test_nodes = np.where(extended_conf_labels == -1)[0]

In [ ]:
len(test_nodes)

In [ ]:
extended_conf_labels[1:100]

In [ ]:
dataset  = torch.utils.data.TensorDataset(torch.tensor(X_extended), torch.tensor(extended_conf_labels))
dataloader = torch.utils.data.DataLoader(dataset, batch_size=50, shuffle=True)

test_dataset  = torch.utils.data.TensorDataset(torch.tensor(X_extended), torch.tensor(extended_real_y))
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=50, shuffle=False)

In [ ]:
m = GCNModel("configs/2_40.txt", 2, dropout=0.0)
m.train(dataloader, 150)

In [ ]:
m.validation_metrics(test_dataloader, train_nodes, test_nodes)